
# Roxy notebook example: CTD descriptors

This notebook is a **reference implementation example** for the **CTD descriptor family** in Roxy.

CTD stands for:

- **Composition**
- **Transition**
- **Distribution**

These descriptors are classical and highly useful because they summarize how residue groups associated with a physicochemical property are distributed along the sequence.

## Covered properties in this notebook

This notebook implements CTD descriptors for:

- hydrophobicity
- polarity
- charge

For each property, the notebook computes:

- **Composition**: fraction of residues in each class
- **Transition**: frequency of transitions between classes
- **Distribution**: normalized positions for the 1st, 25%, 50%, 75%, and 100% occurrence of each class

The notebook is written as a **clean teaching implementation** so it can later be migrated into the real Roxy package.


In [1]:

import math
from collections import Counter

import numpy as np
import pandas as pd


## Demo dataset

In [2]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "ctd_1",
            "ctd_2",
            "ctd_3",
            "ctd_4",
            "ctd_5",
            "ctd_6",
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
            "PPPPGSSSSSTTTTNNQQQ",
            "MSTNPKPQRITLKDGNKVELV",
        ],
        "label": ["A", "B", "A", "B", "A", "B"],
    }
)

df_demo


,sequence_id,sequence,label
0,ctd_1,MKWVTFISLLFLFSSAYSRGVFRR,A
1,ctd_2,GGGGGGGGGGGGGGG,B
2,ctd_3,KRRKRRKRRKRRDDDDEE,A
3,ctd_4,ACDEFGHIKLMNPQRSTVWY,B
4,ctd_5,PPPPGSSSSSTTTTNNQQQ,A
5,ctd_6,MSTNPKPQRITLKDGNKVELV,B


## Constants and CTD class definitions

In [3]:

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

# Three-class partitions for each property.
# These groupings are simple, interpretable, and suitable for a demo implementation.

CTD_GROUPS = {
    "hydrophobicity": {
        "1": set("RKEDQN"),
        "2": set("GASTPHY"),
        "3": set("CLVIMFW"),
    },
    "polarity": {
        "1": set("LIFWCMVY"),
        "2": set("PATGS"),
        "3": set("HQRKNED"),
    },
    "charge": {
        "1": set("KR"),
        "2": set("ANCQGHILMFPSTWYV"),
        "3": set("DE"),
    },
}


## Helper functions

In [4]:

def clean_sequence(seq: str) -> str:
    """Keep only the 20 standard amino acids."""
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper().replace("*", "")
    return "".join([aa for aa in seq if aa in STANDARD_AA])


def assign_ctd_classes(seq: str, property_name: str) -> list[str]:
    """Map sequence residues to CTD class labels ('1', '2', '3')."""
    seq = clean_sequence(seq)
    groups = CTD_GROUPS[property_name]

    class_labels = []
    for aa in seq:
        assigned = None
        for class_id, aa_group in groups.items():
            if aa in aa_group:
                assigned = class_id
                break
        if assigned is not None:
            class_labels.append(assigned)
    return class_labels


def ctd_composition(class_labels: list[str], prefix: str) -> dict:
    """Fraction of residues in each CTD class."""
    total = len(class_labels)
    if total == 0:
        return {f"{prefix}_comp_{class_id}": np.nan for class_id in ("1", "2", "3")}

    counts = Counter(class_labels)
    return {
        f"{prefix}_comp_{class_id}": counts.get(class_id, 0) / total
        for class_id in ("1", "2", "3")
    }


def ctd_transition(class_labels: list[str], prefix: str) -> dict:
    """Normalized frequency of class transitions (12, 13, 23)."""
    transition_keys = [("1", "2"), ("1", "3"), ("2", "3")]

    if len(class_labels) < 2:
        return {
            f"{prefix}_trans_{a}{b}": np.nan
            for a, b in transition_keys
        }

    total_pairs = len(class_labels) - 1
    transition_counts = {("1", "2"): 0, ("1", "3"): 0, ("2", "3"): 0}

    for i in range(total_pairs):
        pair = tuple(sorted((class_labels[i], class_labels[i + 1])))
        if pair in transition_counts:
            transition_counts[pair] += 1

    return {
        f"{prefix}_trans_{a}{b}": transition_counts[(a, b)] / total_pairs
        for a, b in transition_keys
    }


def ctd_distribution(class_labels: list[str], prefix: str) -> dict:
    """Normalized sequence positions for 1st, 25%, 50%, 75%, and 100% class occurrence."""
    percent_labels = ["001", "025", "050", "075", "100"]
    result = {}

    seq_len = len(class_labels)
    if seq_len == 0:
        for class_id in ("1", "2", "3"):
            for p in percent_labels:
                result[f"{prefix}_dist_{class_id}_{p}"] = np.nan
        return result

    for class_id in ("1", "2", "3"):
        positions = [i + 1 for i, label in enumerate(class_labels) if label == class_id]

        if len(positions) == 0:
            for p in percent_labels:
                result[f"{prefix}_dist_{class_id}_{p}"] = np.nan
            continue

        n = len(positions)
        quantile_indices = [
            0,
            math.ceil(0.25 * n) - 1,
            math.ceil(0.50 * n) - 1,
            math.ceil(0.75 * n) - 1,
            n - 1,
        ]

        quantile_positions = [positions[idx] / seq_len for idx in quantile_indices]

        for p, qpos in zip(percent_labels, quantile_positions):
            result[f"{prefix}_dist_{class_id}_{p}"] = qpos

    return result


## Core CTD descriptor function

In [5]:

def ctd_descriptors(seq: str, properties=("hydrophobicity", "polarity", "charge")) -> dict:
    seq = clean_sequence(seq)

    out = {
        "ctd_length": len(seq),
        "ctd_valid_residue_count": len(seq),
    }

    for property_name in properties:
        class_labels = assign_ctd_classes(seq, property_name=property_name)
        prefix = f"ctd_{property_name}"

        out.update(ctd_composition(class_labels, prefix=prefix))
        out.update(ctd_transition(class_labels, prefix=prefix))
        out.update(ctd_distribution(class_labels, prefix=prefix))

    return out


## Functional usage on one sequence

In [6]:

example = ctd_descriptors(df_demo.loc[0, "sequence"])
list(example.items())[:20]


[('ctd_length', 24),
 ('ctd_valid_residue_count', 24),
 ('ctd_hydrophobicity_comp_1', 0.16666666666666666),
 ('ctd_hydrophobicity_comp_2', 0.3333333333333333),
 ('ctd_hydrophobicity_comp_3', 0.5),
 ('ctd_hydrophobicity_trans_12', 0.08695652173913043),
 ('ctd_hydrophobicity_trans_13', 0.13043478260869565),
 ('ctd_hydrophobicity_trans_23', 0.2608695652173913),
 ('ctd_hydrophobicity_dist_1_001', 0.08333333333333333),
 ('ctd_hydrophobicity_dist_1_025', 0.08333333333333333),
 ('ctd_hydrophobicity_dist_1_050', 0.7916666666666666),
 ('ctd_hydrophobicity_dist_1_075', 0.9583333333333334),
 ('ctd_hydrophobicity_dist_1_100', 1.0),
 ('ctd_hydrophobicity_dist_2_001', 0.20833333333333334),
 ('ctd_hydrophobicity_dist_2_025', 0.3333333333333333),
 ('ctd_hydrophobicity_dist_2_050', 0.625),
 ('ctd_hydrophobicity_dist_2_075', 0.7083333333333334),
 ('ctd_hydrophobicity_dist_2_100', 0.8333333333333334),
 ('ctd_hydrophobicity_dist_3_001', 0.041666666666666664),
 ('ctd_hydrophobicity_dist_3_025', 0.166666666

## Inspect class assignment for one property

In [7]:

seq_example = df_demo.loc[0, "sequence"]
assign_ctd_classes(seq_example, property_name="hydrophobicity")[:20]


['3',
 '1',
 '3',
 '3',
 '2',
 '3',
 '3',
 '2',
 '3',
 '3',
 '3',
 '3',
 '3',
 '2',
 '2',
 '2',
 '2',
 '2',
 '1',
 '2']

## Apply CTD descriptors to the full dataset

In [8]:

df_ctd = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(ctd_descriptors).apply(pd.Series),
    ],
    axis=1,
)

df_ctd.head()


,sequence_id,sequence,label,ctd_length,ctd_valid_residue_count,ctd_hydrophobicity_comp_1,ctd_hydrophobicity_comp_2,ctd_hydrophobicity_comp_3,ctd_hydrophobicity_trans_12,ctd_hydrophobicity_trans_13,...,ctd_charge_dist_2_001,ctd_charge_dist_2_025,ctd_charge_dist_2_050,ctd_charge_dist_2_075,ctd_charge_dist_2_100,ctd_charge_dist_3_001,ctd_charge_dist_3_025,ctd_charge_dist_3_050,ctd_charge_dist_3_075,ctd_charge_dist_3_100
0,ctd_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24.0,24.0,0.166667,0.333333,0.50,0.086957,0.130435,...,0.041667,0.250000,0.458333,0.666667,0.916667,NaN,NaN,NaN,NaN,NaN
1,ctd_2,GGGGGGGGGGGGGGG,B,15.0,15.0,0.000000,1.000000,0.00,0.000000,0.000000,...,0.066667,0.266667,0.533333,0.800000,1.000000,NaN,NaN,NaN,NaN,NaN
2,ctd_3,KRRKRRKRRKRRDDDDEE,A,18.0,18.0,1.000000,0.000000,0.00,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,0.722222,0.777778,0.833333,0.944444,1.0
3,ctd_4,ACDEFGHIKLMNPQRSTVWY,B,20.0,20.0,0.300000,0.350000,0.35,0.157895,0.263158,...,0.050000,0.300000,0.550000,0.800000,1.000000,0.150000,0.150000,0.150000,0.200000,0.2
4,ctd_5,PPPPGSSSSSTTTTNNQQQ,A,19.0,19.0,0.263158,0.736842,0.00,0.055556,0.000000,...,0.052632,0.263158,0.526316,0.789474,1.000000,NaN,NaN,NaN,NaN,NaN


## Inspect CTD descriptor groups

In [9]:

hydro_cols = [c for c in df_ctd.columns if c.startswith("ctd_hydrophobicity_")]
polarity_cols = [c for c in df_ctd.columns if c.startswith("ctd_polarity_")]
charge_cols = [c for c in df_ctd.columns if c.startswith("ctd_charge_")]

len(hydro_cols), len(polarity_cols), len(charge_cols)


(21, 21, 21)

In [10]:

df_ctd[
    [
        "sequence_id",
        "ctd_hydrophobicity_comp_1",
        "ctd_hydrophobicity_comp_2",
        "ctd_hydrophobicity_comp_3",
        "ctd_charge_trans_12",
        "ctd_charge_trans_13",
        "ctd_charge_trans_23",
    ]
]


,sequence_id,ctd_hydrophobicity_comp_1,ctd_hydrophobicity_comp_2,ctd_hydrophobicity_comp_3,ctd_charge_trans_12,ctd_charge_trans_13,ctd_charge_trans_23
0,ctd_1,0.166667,0.333333,0.500000,0.217391,0.000000,0.000000
1,ctd_2,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
2,ctd_3,1.000000,0.000000,0.000000,0.000000,0.058824,0.000000
3,ctd_4,0.300000,0.350000,0.350000,0.210526,0.000000,0.105263
4,ctd_5,0.263158,0.736842,0.000000,0.000000,0.000000,0.000000
5,ctd_6,0.428571,0.285714,0.285714,0.350000,0.050000,0.150000


## Dataset-level summary

In [11]:

ctd_cols = [c for c in df_ctd.columns if c.startswith("ctd_") and c not in {"ctd_length", "ctd_valid_residue_count"}]

ctd_summary = (
    df_ctd[ctd_cols]
    .mean(axis=0)
    .sort_values(ascending=False)
    .rename("mean_value")
    .reset_index()
    .rename(columns={"index": "descriptor"})
)

ctd_summary.head(15)


,descriptor,mean_value
0,ctd_charge_dist_2_100,0.983333
1,ctd_polarity_dist_1_100,0.972222
2,ctd_hydrophobicity_dist_3_100,0.955556
3,ctd_hydrophobicity_dist_1_100,0.930952
4,ctd_polarity_dist_3_100,0.930952
5,ctd_hydrophobicity_dist_2_100,0.856892
6,ctd_hydrophobicity_dist_1_075,0.829077
7,ctd_polarity_dist_3_075,0.829077
8,ctd_polarity_dist_2_100,0.826892
9,ctd_charge_dist_1_100,0.806548


## Sanity checks

In [12]:

assert "ctd_hydrophobicity_comp_1" in df_ctd.columns
assert "ctd_polarity_trans_12" in df_ctd.columns
assert "ctd_charge_dist_3_100" in df_ctd.columns
assert df_ctd["ctd_length"].min() > 0

# Composition of three classes should sum to 1 for valid sequences
comp_sum = (
    df_ctd["ctd_hydrophobicity_comp_1"] +
    df_ctd["ctd_hydrophobicity_comp_2"] +
    df_ctd["ctd_hydrophobicity_comp_3"]
)
assert np.allclose(comp_sum, 1.0)

print("CTD descriptor checks passed.")
print(f"Number of hydrophobicity CTD descriptors: {len(hydro_cols)}")
print(f"Number of polarity CTD descriptors: {len(polarity_cols)}")
print(f"Number of charge CTD descriptors: {len(charge_cols)}")


CTD descriptor checks passed.
Number of hydrophobicity CTD descriptors: 21
Number of polarity CTD descriptors: 21
Number of charge CTD descriptors: 21


## Class-style implementation closer to the real package

In [13]:

class CTDDescriptors:
    """Example class-style CTD implementation for later migration into Roxy."""

    def __init__(self, properties=("hydrophobicity", "polarity", "charge")):
        self.properties = properties

    def transform_sequence(self, seq: str) -> dict:
        return ctd_descriptors(seq, properties=self.properties)

    def transform(self, sequences) -> pd.DataFrame:
        return pd.DataFrame([self.transform_sequence(seq) for seq in sequences])


ctd_transformer = CTDDescriptors(properties=("hydrophobicity", "polarity", "charge"))
ctd_matrix = ctd_transformer.transform(df_demo["sequence"].tolist())
ctd_matrix.head()


,ctd_length,ctd_valid_residue_count,ctd_hydrophobicity_comp_1,ctd_hydrophobicity_comp_2,ctd_hydrophobicity_comp_3,ctd_hydrophobicity_trans_12,ctd_hydrophobicity_trans_13,ctd_hydrophobicity_trans_23,ctd_hydrophobicity_dist_1_001,ctd_hydrophobicity_dist_1_025,...,ctd_charge_dist_2_001,ctd_charge_dist_2_025,ctd_charge_dist_2_050,ctd_charge_dist_2_075,ctd_charge_dist_2_100,ctd_charge_dist_3_001,ctd_charge_dist_3_025,ctd_charge_dist_3_050,ctd_charge_dist_3_075,ctd_charge_dist_3_100
0,24,24,0.166667,0.333333,0.50,0.086957,0.130435,0.260870,0.083333,0.083333,...,0.041667,0.250000,0.458333,0.666667,0.916667,NaN,NaN,NaN,NaN,NaN
1,15,15,0.000000,1.000000,0.00,0.000000,0.000000,0.000000,NaN,NaN,...,0.066667,0.266667,0.533333,0.800000,1.000000,NaN,NaN,NaN,NaN,NaN
2,18,18,1.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.055556,0.277778,...,NaN,NaN,NaN,NaN,NaN,0.722222,0.777778,0.833333,0.944444,1.0
3,20,20,0.300000,0.350000,0.35,0.157895,0.263158,0.263158,0.150000,0.200000,...,0.050000,0.300000,0.550000,0.800000,1.000000,0.150000,0.150000,0.150000,0.200000,0.2
4,19,19,0.263158,0.736842,0.00,0.055556,0.000000,0.000000,0.789474,0.842105,...,0.052632,0.263158,0.526316,0.789474,1.000000,NaN,NaN,NaN,NaN,NaN


## Merge transformer output back to the dataset

In [14]:

df_ctd_class = pd.concat([df_demo, ctd_matrix], axis=1)
df_ctd_class.head()


,sequence_id,sequence,label,ctd_length,ctd_valid_residue_count,ctd_hydrophobicity_comp_1,ctd_hydrophobicity_comp_2,ctd_hydrophobicity_comp_3,ctd_hydrophobicity_trans_12,ctd_hydrophobicity_trans_13,...,ctd_charge_dist_2_001,ctd_charge_dist_2_025,ctd_charge_dist_2_050,ctd_charge_dist_2_075,ctd_charge_dist_2_100,ctd_charge_dist_3_001,ctd_charge_dist_3_025,ctd_charge_dist_3_050,ctd_charge_dist_3_075,ctd_charge_dist_3_100
0,ctd_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24,24,0.166667,0.333333,0.50,0.086957,0.130435,...,0.041667,0.250000,0.458333,0.666667,0.916667,NaN,NaN,NaN,NaN,NaN
1,ctd_2,GGGGGGGGGGGGGGG,B,15,15,0.000000,1.000000,0.00,0.000000,0.000000,...,0.066667,0.266667,0.533333,0.800000,1.000000,NaN,NaN,NaN,NaN,NaN
2,ctd_3,KRRKRRKRRKRRDDDDEE,A,18,18,1.000000,0.000000,0.00,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,0.722222,0.777778,0.833333,0.944444,1.0
3,ctd_4,ACDEFGHIKLMNPQRSTVWY,B,20,20,0.300000,0.350000,0.35,0.157895,0.263158,...,0.050000,0.300000,0.550000,0.800000,1.000000,0.150000,0.150000,0.150000,0.200000,0.2
4,ctd_5,PPPPGSSSSSTTTTNNQQQ,A,19,19,0.263158,0.736842,0.00,0.055556,0.000000,...,0.052632,0.263158,0.526316,0.789474,1.000000,NaN,NaN,NaN,NaN,NaN



## Suggested next refactor into the package

A clean migration path into Roxy would be:

- move CTD class definitions into `roxy/core/constants.py`
- move CTD helper logic into `roxy/sequence/ctd.py`
- expose a class such as `CTDDescriptors`
- allow configurable:
  - property selection
  - custom 3-class partitions
  - composition/transition/distribution subsets
- add tests for:
  - empty sequences
  - sequences shorter than 2 residues
  - sequences strongly enriched in a single CTD class
  - lower-case input
  - invalid characters removed during cleaning


## Optional export

In [ ]:
# df_ctd.to_csv("demo_ctd_descriptors.csv", index=False)
